In [1]:
import os
from collections import Counter

In [2]:
file_extensions = []

def getFolderFileCounts(filepath:str, print_on:bool = False, depth:int = 0):
	for next_step in os.listdir(filepath):
		next_path = os.path.join(filepath, next_step)
		if print_on: print(f"|{depth*'-'}{next_step}")
		if os.path.isdir(next_path):
			getFolderFileCounts(next_path, print_on, depth+1)
		else:
			file_extensions.append(next_step.split('.')[-1].lower())
			
getFolderFileCounts("../data/N338")
print(Counter(file_extensions))

Counter({'jpg': 1221, 'pdf': 253, 'db': 17, 'msg': 9, 'doc': 2, 'zip': 2, 'gitignore': 1, 'txt': 1})


In [3]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import torch

# processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
# model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')

MODEL_PATH = "zai-org/GLM-OCR"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)


c:\Users\Milan\Documents\Gelderland_structures\AML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 510/510 [00:01<00:00, 326.20it/s]


In [5]:
from class_definitions.archiveDocument import load_pdfPlumber_pdf
from doc_processing.text_module import extract_pdf_text
from doc_processing.ocr_module import get_page_images

def processSingleArchiveFolder(filePath:str):
	'''
	given a filePath holding an archive folder with possibly a DOC and/or FOTO folder,
	process it to the expected data formats.
	'''

	for next_step in os.listdir(filePath)[:1]:
		next_path = os.path.join(filePath, next_step)

		if os.path.isdir(next_path):
			processSingleArchiveFolder(next_path)
		else:
			file_extension = next_step.split('.')[-1].lower()
			if file_extension == 'pdf':
				pdf_object = load_pdfPlumber_pdf(next_path)
				pdf_texts = extract_pdf_text(pdf_object)
				pdf_pages_images = get_page_images(pdf_object)

				print(pdf_pages_images)
				
				for y, page_chunks in enumerate(pdf_pages_images[0]):
					for x, chunk in enumerate(page_chunks):


						chunk.save(f"../images/{y}_{x}.png")

						messages = [
							{
								"role": "user",
								"content": [
									{"type": "image"},
									{"type": "text", "text": "Text Recognition:"}
								]
							}
						]

						prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

						inputs = processor(images=chunk, text=prompt, return_tensors="pt").to("cuda")

						output = model.generate(
							**inputs,
							max_new_tokens=1024,
							do_sample=False,
							use_cache=True
						)
						output_text = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

						print(f"{y}_{x}:\n{output_text}\n---------------------------------------------------\n")

				pdf_object.close()			
			
processSingleArchiveFolder("../data/N338/338051")

{'Author': 'anonymous', 'CreationDate': 'D:200812161346', 'Producer': '', 'OceScanCompression': '7', 'OceScanModel': 'TCS400', 'OceScanResolution': '150', 'OceScanImageLogic': '1', 'OceImageTagOrientation': '1'}
[[[<PIL.Image.Image image mode=RGB size=512x512 at 0x22212A356D0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A35AD0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A35F50>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A35750>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A358D0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A35B50>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A35CD0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A37CD0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A362D0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A366D0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A37ED0>, <PIL.Image.Image image mode=RGB size=512x512 at 0x22212A36150>, <PIL.Image.Image 